# Line Sensor Calibration & Test

Tests the physical **4x IR line sensor array** (`infrared_lib` / `line_follower_lib`) —
not the camera. TurboPi has two separate line-following paths:
- `myRobot.line.use_pid()` — this one, PID steering from the 4 IR sensors (default)
- `myRobot.line.use_camera()` — a separate ROS node that follows a line with the camera

## What "calibration" means for this hardware

The sensor board returns **booleans only** (`True`/`False` per channel) — each channel's
light/dark threshold is set by a physical potentiometer on the sensor board itself, not
in software. There's no `calibrate()` call like the camera's colour calibration. What you
*can* tune from here:

1. **I2C address** — hardware ships at `0x78`, but some boards are wired at `0x77`
   instead. If every read comes back `[False, False, False, False]` regardless of
   surface, check the address first.
2. **Mounting height / sensor spacing vs track width** — mechanical, checked visually below.
3. **PID gains** (`kp`, `ki`, `kd`), `base_speed`, and sensor `weights` — the actual
   "calibration" that matters for smooth line following.

## A note on intermittent I2C errors

`line_sensors.py` itself warns about this: `OSError: [Errno 121] Remote I/O error` shows
up occasionally on this board even when everything is wired correctly — it's known
bus flakiness, not necessarily a fault. Every read in this notebook retries several times
and reports rather than crashes on an isolated failure. A handful of failures over a
12-second test is normal; if **every** read fails, that's wiring/address, not flakiness.

Run the cells in order. Put the robot on a bench with wheels clear of the ground for the
first two sections — nothing drives until the PID test section at the bottom.

## 1. Check the I2C address

If this prints `0x78` in the device list, the default address is correct — skip to section 2.
If not, note whichever address *does* show up and pass it to `get_infrared(address=...)` below.

In [ ]:
import infrared_lib

ir = infrared_lib.get_infrared()  # default address 0x78
print("Scanning I2C bus...")
devices = ir.scan_i2c_bus()
print("Devices found:", devices)
print("Expecting the line sensor at 0x78 (or 0x77 on some boards).")


## 2. Live raw sensor read

Slide a strip of the actual track (black line on white, or white line on black —
whatever your track uses) under the sensor by hand while this runs. Watch which of the
4 booleans flips as the line passes under each sensor.

Sensor order is `[s0, s1, s2, s3]`, left to right. If the order looks reversed on your
robot, that's useful to know — `LineFollower(weights=...)` below can be given
`[3, 1, -1, -3]` instead of the default to flip it in software rather than remounting
anything.

In [ ]:
import time

READ_SECONDS = 12
INTERVAL_S = 0.2
READ_RETRIES = 10       # this sensor board is known to drop I2C reads intermittently
READ_RETRY_DELAY_S = 0.05

print(f"Reading for {READ_SECONDS}s — move the line under the sensor now...")
end = time.time() + READ_SECONDS
last = None
errors = 0
while time.time() < end:
    try:
        states = ir.read(retries=READ_RETRIES, retry_delay_s=READ_RETRY_DELAY_S)
    except OSError as e:
        errors += 1
        print(f"  (I2C read failed after {READ_RETRIES} retries: {e} — skipping this sample)")
        time.sleep(INTERVAL_S)
        continue
    if states != last:
        print(f"[s0 s1 s2 s3] = {[int(s) for s in states]}")
        last = states
    time.sleep(INTERVAL_S)
print("Done. Every channel should have flipped True at least once as the line passed under it.")
print("If a channel never changed: check its height above the track and the onboard")
print("potentiometer on the sensor board (turn until it reliably distinguishes line vs floor).")
if errors:
    print(f"\n{errors} read(s) failed even after {READ_RETRIES} retries. A handful over 12s is normal")
    print("I2C flakiness for this board (see the comment in line_sensors.py). If EVERY read fails,")
    print("that's wiring/address, not flakiness — recheck section 1.")


## 3. Preview the steering error

Before driving, sanity-check the direction sense: a positive error should mean the line is to the *right* of centre (robot should turn right to correct), negative means left.

Hold the line under the left-most sensor, then the right-most, and confirm the sign matches.

In [ ]:
WEIGHTS = [-3.15, -0.75, 0.75, 3.15]  # measured offsets (cm) from array centre; flip order if sensor order is reversed

def preview_error(states, weights=WEIGHTS):
    return sum(w * int(bool(s)) for w, s in zip(weights, states))

try:
    states = ir.read(retries=10, retry_delay_s=0.05)
except OSError as e:
    print(f"I2C read failed even after retries: {e} — try re-running this cell.")
else:
    error = preview_error(states)
    print(f"states={[int(s) for s in states]}  error={error:+.1f}  "
          f"({'line to the right — should turn right' if error > 0 else 'line to the left — should turn left' if error < 0 else 'centred or no line'})")


## 4. PID line-following test

Now the robot actually drives. Put it on the track, clear the area, and run the next cell.

Start conservative (`base_speed=150-200`, default gains) and increase once it tracks reliably. If it oscillates side-to-side, lower `kp` or raise `kd`. If it responds too slowly to curves, raise `kp`.

`follow_for()` stops automatically at a T-junction (all 4 sensors on) unless you pass `stop_at_junction=False`.

In [ ]:
import line_follower_lib as llf

pid = llf.PIDConfig(kp=25.0, ki=0.0, kd=4.0)  # tune these
follower = llf.LineFollower(
    infrared=ir,
    base_speed=180.0,      # start low, raise once it's tracking well
    weights=WEIGHTS,
    pid=pid,
    max_turn=0.8,
    junction_action="stop",
)

print("Starting in 2s — make sure the track is clear and the robot is on the line...")
time.sleep(2)
try:
    hit_junction = follower.follow_for(duration_s=5.0, step_s=0.05)
    print("Stopped —", "hit a junction" if hit_junction else "timed out (still on the line, or lost it)")
except OSError as e:
    follower.rm.stop()
    print(f"Stopped — I2C read failed mid-drive: {e}. Re-run this cell to try again.")


## 5. Step-by-step debug (optional)

Runs one control step at a time and prints the full debug dict — useful for watching `error`/`turn` react in real time instead of a fixed-duration run. Interrupt the kernel (■) to stop.

In [ ]:
follower.reset()
try:
    while True:
        info = follower.step(seconds=0.05)
        print(f"states={[int(s) for s in info['states']]}  error={info['error']:+.1f}  "
              f"turn={info['turn']:+.2f}  junction={info['junction']}")
        if info["junction"]:
            break
except (KeyboardInterrupt, OSError) as e:
    if isinstance(e, OSError):
        print(f"I2C read failed: {e}")
finally:
    follower.rm.stop()
    print("Stopped.")


## 6. Emergency stop

Run this any time if the robot needs to stop immediately.

In [ ]:
follower.rm.stop()
print("Stopped.")
